In [80]:
import os
import random
import re
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import faiss
import sentence_transformers
import torch
import wikipedia

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)

SEED = 42

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

device: cpu


In [63]:
# ЗАгружаем базу знаний
# Скачиваем статьи с русской Википедии
wikipedia.set_lang("ru")

ai_topics = [
    "Искусственный интеллект",
    "Машинное обучение",
    "Глубокое обучение",
    "Обучение с подкреплением",
    "Генеративный искусственный интеллект",
    "Нейронная сеть",
    "Обучение с учителем",
    "Обучение без учителя",
    "Перцептрон",
    "Обработка естественного языка",
]

# Создаем папку для сохранения статей
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

def download_documents(topics: list[str]) -> List[Dict[str, str]]:
    filenames = [f"{output_dir}/{topic.replace(' ', '_')}.txt" for topic in topics]
    # Скачиваем статьи
    for i, topic in enumerate(topics, 1):
        filename = f"{output_dir}/{topic.replace(' ', '_')}.txt"
        if os.path.exists(filename):
            continue

        try:
            print(f"Скачивание статьи {i}/{len(topics)}: {topic}")
            page = wikipedia.page(topic, auto_suggest=True)
            
            # Сохраняем в файл
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(page.content)
            
            print(f"✓ Сохранено: {filename}")
            
        except wikipedia.exceptions.DisambiguationError as e:
            print(f"⚠ Неоднозначность: {topic} - {e.options[:3]}")
        except wikipedia.exceptions.PageError:
            print(f"✗ Статья не найдена: {topic}")
        except Exception as e:
            print(f"✗ Ошибка при скачивании {topic}: {e}")


    documents: list[Dict[str, str]] = []
    for i, filename in enumerate(filenames, 1):
        with open(filename, 'r', encoding='utf-8') as f:
            content = f.read()
            documents.append({
                "doc_id": f"doc_{i}",
                "title": topics[i-1],
                "text": content
            })
    
    return documents

documents = download_documents(ai_topics)
print(*documents, sep='\n')
print(f"Число документов: {len(documents)}")

{'doc_id': 'doc_1', 'title': 'Искусственный интеллект', 'text': 'Иску́сственный интелле́кт (англ. artificial intelligence; AI), также ИИ, искусственный ра́зум, в самом широком смысле — научно-технологическая область, занимающаяся изучением и созданием интеллектуальных сущностей (англ. intelligent entities), способных «вычислять, как им действовать эффективно и безопасно в самых разнообразных, в том числе незнакомых им, ситуациях», и решать задачи как минимум уровня человеческого интеллекта и реализованных машинами, в частности компьютерными системами. Это направление исследований в области компьютерных наук, которая разрабатывает и изучает методы и программное обеспечение, позволяющие машинам воспринимать окружающую среду и использовать обучение и интеллект для выполнения действий, которые максимально увеличивают их шансы на достижение поставленных целей. Такие машины можно назвать искусственным интеллектом. В то же время не следует путать понятия ИИ и больших языковых моделей (БЯМ/LLM

База знаний представляет собой статьи из русскоязычной Википедии по предметной области "Искусственный интеллект". Некоторые из тем статей пересекаются. По данным статьям можно задать много вопросов, так как текст являются достаточно содержательными.

In [39]:
# Простая функция чанкинга по словам.
def chunk_text(text: str, chunk_size: int = 22, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks

def build_chunks_dataframe(
    docs: List[Dict[str, str]],
    chunk_size: int = 22,
    overlap: int = 5,
) -> pd.DataFrame:
    rows = []

    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk in enumerate(chunks):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": chunk_id,
                    "chunk_text": chunk,
                    "n_words": len(chunk.split()),
                }
            )

    return pd.DataFrame(rows)


chunks_df = build_chunks_dataframe(documents, chunk_size=50, overlap=10)

print("Количество чанков:", len(chunks_df))
display(chunks_df)

Количество чанков: 704


,doc_id,title,chunk_id,chunk_text,n_words
0,doc_1,Искусственный интеллект,0,Иску́сственный интелле́кт (англ. artificial in...,50
1,doc_1,Искусственный интеллект,1,"незнакомых им, ситуациях», и решать задачи как...",50
2,doc_1,Искусственный интеллект,2,"обучение и интеллект для выполнения действий, ...",50
3,doc_1,Искусственный интеллект,3,"компьютерных наук, охватывающая решение широко...",50
4,doc_1,Искусственный интеллект,4,"по сути, специализированным вероятностным алго...",50
...,...,...,...,...,...
699,doc_10,Обработка естественного языка,9,"а кто рыжий, не обойтись (кроме того, что лиса...",50
700,doc_10,Обработка естественного языка,10,"порождает дополнительную проблему, вытекающую ...",50
701,doc_10,Обработка естественного языка,11,системы Генерирование текста Синтез речи Задач...,50
702,doc_10,Обработка естественного языка,12,аннотация Семантическая аннотация Генерировани...,50


Для разбиения текстов на чанки мы выбираем параметры chunk_size и overlap. Chunk_size отвечает за длину чанка, а overlap за наложение соседних чанков друг на друга. Это нужно, чтобы у соседних чанков было немного общего контекста.

In [26]:
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")
    

def build_embedding_backend(
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
) -> EmbeddingBackend:
    backend = SentenceTransformersBackend(model_name=model_name, device=device)
    print("Используем полноценные dense embeddings.")
    print("Бэкэнд:", backend.backend_name)
    return backend


embedder = build_embedding_backend(device=DEVICE)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6554.06it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [27]:
# Строим векторные представления для всех чанков.
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)

print("Форма матрицы эмбеддингов:", chunk_embeddings.shape)

# Проверяем длины векторов.
# Если normalize_embeddings=True сработал корректно, все нормы должны быть ≈ 1.0.
# Это означает, что косинусное сходство далее можно считать через скалярное произведение.
vector_norms = np.linalg.norm(chunk_embeddings, axis=1)
print("Минимальная норма:", round(float(vector_norms.min()), 4))
print("Максимальная норма:", round(float(vector_norms.max()), 4))
print("Средняя норма:", round(float(vector_norms.mean()), 4))
print("→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.")

Форма матрицы эмбеддингов: (704, 384)
Минимальная норма: 1.0
Максимальная норма: 1.0
Средняя норма: 1.0
→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.


In [ ]:
# Ищем похожие вектора через FAISS
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self._nn_index = None
        self._faiss_index = faiss.IndexFlatIP(dim)  # type: ignore[name-defined]

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")

        if self._faiss_index is not None:
            self._faiss_index.add(vectors)
        else:
            self._nn_index.fit(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")

        if self._faiss_index is not None:
            scores, indices = self._faiss_index.search(query_vectors, top_k)
            return scores, indices

        distances, indices = self._nn_index.kneighbors(query_vectors, n_neighbors=top_k)
        scores = 1.0 - distances
        return scores, indices


class Retriever:
    def __init__(self, documents, embedder: EmbeddingBackend, chunk_size=50, overlap=5,  device=DEVICE):
        self.chunks_df = build_chunks_dataframe(documents, chunk_size=chunk_size, overlap=overlap)
        self.embedder = embedder
        self.chunk_texts = self.chunks_df["chunk_text"].tolist()
        self.chunk_embeddings = self.embedder.fit_documents(self.chunk_texts)
        self.search_index = VectorSearchIndex(dim=self.chunk_embeddings.shape[1])
        self.search_index.add(self.chunk_embeddings)
        print("Индекс построен.")

    def search_chunks(self, query: str, top_k: int = 5) -> pd.DataFrame:
        query_vectors = self.embedder.encode_queries([query])
        scores, indices = self.search_index.search(query_vectors, top_k=top_k)

        rows = []
        for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
            chunk_row = self.chunks_df.iloc[int(idx)]
            rows.append(
                {
                    "rank": rank,
                    "doc_id": chunk_row["doc_id"],
                    "title": chunk_row["title"],
                    "chunk_id": int(chunk_row["chunk_id"]),
                    "score": round(float(score), 4),
                    "chunk_text": chunk_row["chunk_text"],
                }
            )

        return pd.DataFrame(rows)
    
    def display_query_results(self, query: str, top_k: int = 5):
        results_df = self.search_chunks(query, top_k=top_k)
        display(Markdown(f"**Запрос:** {query}"))
        display(results_df)


retriever = Retriever(documents, embedder=embedder, chunk_size=50, overlap=5, device=DEVICE)

retriever.display_query_results("Как обучается простая нейросеть?")
retriever.display_query_results("Что такое перцептрон?")
retriever.display_query_results("Какие наиболее популяреные системы искусственного интеллекта существуют сейчас?")

Индекс построен.


**Запрос:** Как обучается простая нейросеть?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_3,Глубокое обучение,48,0.6905,с большими объёмами немаркированных данных. Не...
1,2,doc_3,Глубокое обучение,27,0.6795,"(выходного) слоя, возможно, после многократног..."
2,3,doc_2,Машинное обучение,5,0.6772,любого возможного входного объекта выдать дост...
3,4,doc_9,Перцептрон,56,0.6745,это самый популярный метод обучения многослойн...
4,5,doc_9,Перцептрон,5,0.6721,"статье «Логическое исчисление идей, относящихс..."


**Запрос:** Что такое перцептрон?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_9,Перцептрон,67,0.6391,"целый набор входных значений. Во-вторых, из-за..."
1,2,doc_9,Перцептрон,42,0.6204,ниже). Многослойный перцептрон (по Розенблатту...
2,3,doc_9,Перцептрон,6,0.6033,американский нейрофизиолог Фрэнк Розенблатт. О...
3,4,doc_9,Перцептрон,43,0.5774,обучаемыми являются все слои перцептрона (в то...
4,5,doc_9,Перцептрон,1,0.5578,"моделей нейросетей, а «Марк-1» — первым в мире..."


**Запрос:** Какие наиболее популяреные системы искусственного интеллекта существуют сейчас?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_1,Искусственный интеллект,79,0.7900,которых в формальном виде требует значительных...
1,2,doc_1,Искусственный интеллект,83,0.7685,"единую систему, способную решать проблемы чело..."
2,3,doc_1,Искусственный интеллект,5,0.7535,а также сверхчеловеческую игру и анализ в стра...
3,4,doc_1,Искусственный интеллект,0,0.7528,Иску́сственный интелле́кт (англ. artificial in...
4,5,doc_1,Искусственный интеллект,3,0.7441,и самый популярный в середине 2020-х) из многи...


In [52]:
# Вспомогательные функции для benchmark

def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered


def evaluate_query(retriever: Retriever, query: str, relevant_doc_ids: List[str], top_k: int = 3) -> Dict[str, object]:
    result_df = retriever.search_chunks(query, top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)

    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)

    first_relevant_rank = None
    for idx, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            first_relevant_rank = idx
            break

    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank

    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }


def evaluate_benchmark(benchmark_rows: List[Dict[str, str]], retriever: Retriever, top_k: int = 3) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(
            retriever=retriever,
            query=row["query"],
            relevant_doc_ids=row["relevant_doc_ids"],
            top_k=top_k,
        )
        rows.append(
            {
                "query_id": row["query_id"],
                "query": row["query"],
                "expected_source": ", ".join(row["relevant_doc_ids"]),
                "retrieved_sources": ", ".join(metrics["predicted_doc_ids"]),
                f"hit_at_k": metrics["hit"],
                f"recall_at_k": metrics["recall"],
                f"MRR_at_k": metrics["mrr"],
                "rank_of_first_relevant": metrics["first_relevant_rank"],
            }
        )
    return pd.DataFrame(rows)

In [54]:
benchmark_rows = [
    {
        "query_id": "q01",
        "query": "Что такое искусственный интеллект?",
        "relevant_doc_ids": ["doc_1", "doc_2"]
    },
    {
        "query_id": "q02",
        "query": "Какие бывают методы машинного обучения?",
        "relevant_doc_ids": ["doc_2", "doc_7", "doc_8", "doc_4"]
    },
    {
        "query_id": "q03",
        "query": "Как работает нейронная сеть?",
        "relevant_doc_ids": ["doc_6", "doc_3"]
    },
    {
        "query_id": "q04",
        "query": "Что такое обучение с подкреплением?",
        "relevant_doc_ids": ["doc_4"]
    },
    {
        "query_id": "q05",
        "query": "Генеративный искусственный интеллект и его возможности",
        "relevant_doc_ids": ["doc_5"]
    },
    {
        "query_id": "q06",
        "query": "В чем различие между обучением с учителем и без учителя?",
        "relevant_doc_ids": ["doc_7", "doc_8"]
    },
    {
        "query_id": "q07",
        "query": "Что такое перцептрон?",
        "relevant_doc_ids": ["doc_9", "doc_6"]
    },
    {
        "query_id": "q08",
        "query": "В каких областях применяется генеративный ИИ?",
        "relevant_doc_ids": ["doc_5"]
    },
    {
        "query_id": "q09",
        "query": "Какие модели используются для обработки естественного языка?",
        "relevant_doc_ids": ["doc_10"]
    }
]

benchmark_result = evaluate_benchmark(benchmark_rows, retriever)
display(benchmark_result)
benchmark_result.to_csv("artifacts/retrieval_eval.csv")

,query_id,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,MRR_at_k,rank_of_first_relevant
0,q01,Что такое искусственный интеллект?,"doc_1, doc_2",doc_1,1,0.50,1.000000,1
1,q02,Какие бывают методы машинного обучения?,"doc_2, doc_7, doc_8, doc_4",doc_2,1,0.25,1.000000,1
2,q03,Как работает нейронная сеть?,"doc_6, doc_3",doc_6,1,0.50,1.000000,1
3,q04,Что такое обучение с подкреплением?,doc_4,doc_4,1,1.00,1.000000,1
4,q05,Генеративный искусственный интеллект и его воз...,doc_5,"doc_1, doc_5",1,1.00,0.500000,2
5,q06,В чем различие между обучением с учителем и бе...,"doc_7, doc_8","doc_8, doc_4, doc_6",1,0.50,1.000000,1
6,q07,Что такое перцептрон?,"doc_9, doc_6",doc_9,1,0.50,1.000000,1
7,q08,В каких областях применяется генеративный ИИ?,doc_5,"doc_1, doc_5",1,1.00,0.500000,2
8,q09,Какие модели используются для обработки естест...,doc_10,"doc_5, doc_1, doc_10",1,1.00,0.333333,3


In [59]:
retriever2 = Retriever(documents, embedder=embedder, chunk_size=100, overlap=5, device=DEVICE)
benchmark_result2 = evaluate_benchmark(benchmark_rows, retriever2)
display(benchmark_result2)

Индекс построен.


,query_id,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,MRR_at_k,rank_of_first_relevant
0,q01,Что такое искусственный интеллект?,"doc_1, doc_2",doc_1,1,0.5,1.000000,1.0
1,q02,Какие бывают методы машинного обучения?,"doc_2, doc_7, doc_8, doc_4","doc_2, doc_7, doc_1",1,0.5,1.000000,1.0
2,q03,Как работает нейронная сеть?,"doc_6, doc_3","doc_6, doc_9",1,0.5,1.000000,1.0
3,q04,Что такое обучение с подкреплением?,doc_4,"doc_4, doc_7, doc_2",1,1.0,1.000000,1.0
4,q05,Генеративный искусственный интеллект и его воз...,doc_5,doc_1,0,0.0,0.000000,NaN
5,q06,В чем различие между обучением с учителем и бе...,"doc_7, doc_8","doc_6, doc_8",1,0.5,0.500000,2.0
6,q07,Что такое перцептрон?,"doc_9, doc_6",doc_9,1,0.5,1.000000,1.0
7,q08,В каких областях применяется генеративный ИИ?,doc_5,"doc_5, doc_1",1,1.0,1.000000,1.0
8,q09,Какие модели используются для обработки естест...,doc_10,"doc_5, doc_1, doc_10",1,1.0,0.333333,3.0


При изменении chunk_size с 50 на 100, retriever неправильно нашел релевантный документ для запроса q05. Это показывает, что при большом размере чанка контекст становиться менее точным.

In [ ]:
# расширяем базу знаний новыми документами
new_topics = [
    "Большая языковая модель",
    "Трансформер (модель машинного обучения)",
    "Generative pre-trained transformer",
]

extended_topics = ai_topics + new_topics
extended_documents = download_documents(extended_topics)
print(*extended_documents, sep='\n')

Скачивание статьи 11/13: Большая языковая модель
✓ Сохранено: data/Большая_языковая_модель.txt
Скачивание статьи 12/13: Трансформер (модель машинного обучения)
✓ Сохранено: data/Трансформер_(модель_машинного_обучения).txt
Скачивание статьи 13/13: Generative pre-trained transformer
✓ Сохранено: data/Generative_pre-trained_transformer.txt
{'doc_id': 'doc_1', 'title': 'Искусственный интеллект', 'text': 'Иску́сственный интелле́кт (англ. artificial intelligence; AI), также ИИ, искусственный ра́зум, в самом широком смысле — научно-технологическая область, занимающаяся изучением и созданием интеллектуальных сущностей (англ. intelligent entities), способных «вычислять, как им действовать эффективно и безопасно в самых разнообразных, в том числе незнакомых им, ситуациях», и решать задачи как минимум уровня человеческого интеллекта и реализованных машинами, в частности компьютерными системами. Это направление исследований в области компьютерных наук, которая разрабатывает и изучает методы и прог

In [67]:
new_query_rows = [
    {
        "query_id": "q05",
        "query": "Генеративный искусственный интеллект и его возможности",
        "relevant_doc_ids": ["doc_5"]
    },
    {
        "query_id": "q11",
        "query": "Когда была выпущена первая модель GPT",
        "relevant_doc_ids": ["doc_12"]
    },
    {
        "query_id": "q12",
        "query": "Из чего состоит трансформер?",
        "relevant_doc_ids": ["doc_12"]
    },
]

retriever_extended = Retriever(extended_documents, embedder=embedder, chunk_size=50, overlap=5, device=DEVICE)


display(Markdown(f"**До обновления базы знаний:**"))
for row in new_query_rows:
    retriever.display_query_results(row["query"])
display(Markdown(f"**После обновления базы знаний:**"))
for row in new_query_rows:
    retriever_extended.display_query_results(row["query"])


Индекс построен.


**До обновления базы знаний:**

**Запрос:** Генеративный искусственный интеллект и его возможности

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_1,Искусственный интеллект,110,0.8144,=== Применение искусственного интеллекта являе...
1,2,doc_5,Генеративный искусственный интеллект,0,0.7950,Генеративный искусственный интеллект — тип сис...
2,3,doc_1,Искусственный интеллект,0,0.7874,Иску́сственный интелле́кт (англ. artificial in...
3,4,doc_1,Искусственный интеллект,197,0.7800,искусственный интеллект Искусственный интеллек...
4,5,doc_1,Искусственный интеллект,88,0.7709,«игровой искусственный интеллект». Стандартным...


**Запрос:** Когда была выпущена первая модель GPT

...

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_1,Искусственный интеллект,110,0.8144,=== Применение искусственного интеллекта являе...
1,2,doc_5,Генеративный искусственный интеллект,0,0.7950,Генеративный искусственный интеллект — тип сис...
2,3,doc_1,Искусственный интеллект,0,0.7874,Иску́сственный интелле́кт (англ. artificial in...
3,4,doc_1,Искусственный интеллект,197,0.7800,искусственный интеллект Искусственный интеллек...
4,5,doc_1,Искусственный интеллект,88,0.7709,«игровой искусственный интеллект». Стандартным...


**Запрос:** Когда была выпущена первая модель GPT

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_11,Большая языковая модель,13,0.7686,История развития моделей GPT отражает стремите...
1,2,doc_11,Большая языковая модель,12,0.6066,году была представлена модель BERT (encoder-on...
2,3,doc_12,Трансформер (модель машинного обучения),9,0.5753,"как GPT. В частности, на основе GPT версии 3.5..."
3,4,doc_13,Generative pre-trained transformer,5,0.5624,дискриминативное (различительное) «дообучающее...
4,5,doc_11,Большая языковая модель,14,0.5570,API без возможности локального запуска. Настоя...


**Запрос:** Из чего состоит трансформер?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_12,Трансформер (модель машинного обучения),0,0.6310,Трансфо́рмер (англ. Transformer) — архитектура...
1,2,doc_12,Трансформер (модель машинного обучения),1,0.5016,"на естественном языке, и решения таких задач к..."
2,3,doc_11,Большая языковая модель,35,0.4424,трансформера (энкодер-декодер) комбинируют пре...
3,4,doc_13,Generative pre-trained transformer,0,0.4384,Generative pre-trained transformer или GPT (с ...
4,5,doc_11,Большая языковая модель,11,0.3939,не существовали. В 2017 году на конференции Ne...


По результатам запросов видно, что после добавления новых документов retriever стал опираться на них при специфических запросах.

In [76]:
def retrieval_comparison(benchmark_result1, benchmark_result2) -> pd.DataFrame:
    sources1 = benchmark_result1["retrieved_sources"].to_list()
    sources2 = benchmark_result2["retrieved_sources"].to_list()

    comparison = pd.DataFrame(
        {
            "query": benchmark_result1["query"],
            "before_retrieved_sources": sources1,
            "after_retrieved_sources": sources2,
            "changed": "",
        }
    )
    return comparison

result_before_update = evaluate_benchmark(new_query_rows, retriever)
result_after_update = evaluate_benchmark(new_query_rows, retriever_extended)
comparison_df = retrieval_comparison(result_before_update, result_after_update)
display(comparison_df)
comparison_df.to_csv("artifacts/retrieval_before_after_update.csv")

,query,before_retrieved_sources,after_retrieved_sources,changed
0,Генеративный искусственный интеллект и его воз...,"doc_1, doc_5","doc_1, doc_5",
1,Когда была выпущена первая модель GPT,"doc_2, doc_5, doc_6","doc_11, doc_12",
2,Из чего состоит трансформер?,"doc_9, doc_1","doc_12, doc_11",


In [77]:
# Создаем мини-RAG
def build_context_from_retrieval(query: str, retriever: Retriever, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = retriever.search_chunks(query, top_k=top_k)
    context_blocks = []

    for _, row in retrieved.iterrows():
        block = (
            f"[Источник: {row['doc_id']} | {row['title']} | score={row['score']:.4f}]\n"
            f"{row['chunk_text']}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, retrieved

In [83]:
query = "Из каких компонентов состоит трансформер?"
context, retrieved_df = build_context_from_retrieval(query, retriever_extended, top_k=3)

display(Markdown(f"### Запрос: {query}"))
display(retrieved_df)
print(context)

### Запрос: Из каких компонентов состоит трансформер?

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_12,Трансформер (модель машинного обучения),0,0.6112,Трансфо́рмер (англ. Transformer) — архитектура...
1,2,doc_12,Трансформер (модель машинного обучения),1,0.4868,"на естественном языке, и решения таких задач к..."
2,3,doc_11,Большая языковая модель,35,0.4758,трансформера (энкодер-декодер) комбинируют пре...


[Источник: doc_12 | Трансформер (модель машинного обучения) | score=0.6112]
Трансфо́рмер (англ. Transformer) — архитектура глубоких нейронных сетей, представленная в 2017 году исследователями из Google Brain / Google Research в статье «Внимание — это всё, что вам нужно». По аналогии с рекуррентными нейронными сетями (РНС / RNN) трансформеры предназначены для обработки последовательностей, таких как текст на естественном языке, и решения

[Источник: doc_12 | Трансформер (модель машинного обучения) | score=0.4868]
на естественном языке, и решения таких задач как машинный перевод и автоматическое реферирование. В отличие от РНС, трансформеры не требуют обработки последовательностей по порядку. Например, если входные данные — это текст, то трансформеру не требуется обрабатывать конец текста после обработки его начала. Благодаря этому трансформеры распараллеливаются легче чем РНС и

[Источник: doc_11 | Большая языковая модель | score=0.4758]
трансформера (энкодер-декодер) комбинируют преиму

In [81]:
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    # Убираем технические строки источников из ранжирования, но не из общего контекста.
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 4]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."

    return " ".join(selected_sentences)

In [84]:
answer_example = generate_answer_from_context(query, context)
print(answer_example)

Transformer) — архитектура глубоких нейронных сетей, представленная в 2017 году исследователями из Google Brain / Google Research в статье «Внимание — это всё, что вам нужно». трансформера (энкодер-декодер) комбинируют преимущества обоих компонентов: энкодер обрабатывает все входные элементы, а декодер генерирует выходные с доступом к представлениям энкодера, что делает их оптимальными для суммаризации, машинного перевода и генеративных вопросно-ответных систем.


In [107]:
# Упакуем в mini-RAG пайплайн
def mini_rag_answer(
    query: str,
    retriever: Retriever,
    top_k: int = 3,
    max_answer_sentences: int = 2,
) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, retriever=retriever, top_k=top_k)
    answer = generate_answer_from_context(query, context=context, max_sentences=max_answer_sentences)

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "sources": retrieved,
    }


def run_test(queries: List[str], retriever: Retriever, save=True):
    test_results = []

    for query in queries:
        rag_result = mini_rag_answer(query, retriever=retriever)
        display(Markdown(f"### Вопрос: {rag_result['query']}"))
        display(Markdown(f"**Ответ:** {rag_result['answer']}"))
        display(Markdown("**Источники:**"))
        display(rag_result["sources"])

        test_results.append({
            "question": query,
            "answer": rag_result['answer'],
            "retrieved_sources": ", ".join(sorted(list(rag_result["sources"]["doc_id"].unique())))
        })

    test_results_df = pd.DataFrame(test_results)
    if save:
        test_results_df.to_csv("artifacts/rag_examples.csv")

# Покажем примеры работы RAG
test_queries = [
    "Для каких задач используется обучение с подкреплением?",
    "Что такое GPT?",
    "Какие бывают архитектуры нейросетей?",
    "Как работает нейросеть?",
    "Какая сегодня погода"
]

run_test(test_queries, retriever_extended)

### Вопрос: Для каких задач используется обучение с подкреплением?

**Ответ:** Обучение с подкреплением (англ. для задач, в которых известны описания множества объектов (обучающей выборки), и требуется обнаружить внутренние взаимосвязи, зависимости, закономерности, существующие между объектами.

**Источники:**

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_4,Обучение с подкреплением,17,0.6279,изменять с течением времени матрицу взаимодейс...
1,2,doc_8,Обучение без учителя,1,0.6279,"для задач, в которых известны описания множест..."
2,3,doc_4,Обучение с подкреплением,0,0.6207,Обучение с подкреплением (англ. reinforcement ...


### Вопрос: Что такое GPT?

...

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_9,Перцептрон,1,0.5697,"моделей нейросетей, а «Марк-1» — первым в мире..."
1,2,doc_3,Глубокое обучение,26,0.5613,"и 1. Нейроны и синапсы могут также иметь вес, ..."
2,3,doc_6,Нейронная сеть,0,0.5610,Нейро́нная сеть (также иску́сственная нейро́нн...


### Вопрос: Какая сегодня погода

**Ответ:** Проводится следующее преобразование — выстраивается в ряд курс за сегодня, вчера, за позавчера. Обученной сети подаётся на вход курс за сегодня, вчера, позавчера и получается ответ на завтра.

**Источники:**

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,doc_6,Нейронная сеть,77,0.2903,"вчера, за позавчера. Следующий ряд — смещается..."
1,2,doc_6,Нейронная сеть,78,0.1817,дня. Обученной сети подаётся на вход курс за с...
2,3,doc_6,Нейронная сеть,76,0.1806,обучению и обобщению; адаптивность; свойство к...


Как видно из примеров, mini-RAG хорошо справляется с вопросами, ответ на которые напрямую находится в исходных текстах. Если задать более общий вопрос, ответ на который нужно собрать из нескольких документов, то RAG начинает путаться. Для исправления данной проблемы можно попробовать увеличить размер чанка и оверлап. Если задать вопрос, не относящийся к тематике документов, то RAG выдает ответ с похожими по смыслу словами с очень низкой релевантностью.